In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import balanced_accuracy_score
import random
import os
from pretty_print_sklearn_tree import pretty_print_sklearn_tree

DATA_DIR = "./data_product_reviews"

X_train = pd.read_csv(os.path.join(DATA_DIR, "x_train.csv"))
X_valid = pd.read_csv(os.path.join(DATA_DIR, "x_valid.csv"))
X_test  = pd.read_csv(os.path.join(DATA_DIR, "x_test.csv"))

y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))['has_positive_sentiment']
y_valid = pd.read_csv(os.path.join(DATA_DIR, "y_valid.csv"))['has_positive_sentiment']
y_test  = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))['has_positive_sentiment']

feature_names = list(X_train.columns)

print("Data loaded successfully!")

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")

X_all = pd.concat([X_train, X_valid], axis=0)
y_all = pd.concat([y_train, y_valid], axis=0)
test_fold = [-1] * len(X_train) + [0] * len(X_valid)
mysplitter = PredefinedSplit(test_fold)

# Part 2.1 — Simple Decision Tree

dt_simple = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)
dt_simple.fit(X_train, y_train)

# Pretty print ASCII tree
print("\n=== Decision Tree (max_depth=3) ===")
tree = dt_simple
pretty_print_sklearn_tree(tree, feature_names=feature_names)

# Evaluate
train_acc = balanced_accuracy_score(y_train, dt_simple.predict(X_train))
valid_acc = balanced_accuracy_score(y_valid, dt_simple.predict(X_valid))
test_acc  = balanced_accuracy_score(y_test, dt_simple.predict(X_test))

print(f"Balanced Accuracy — Train: {train_acc:.4f}, Valid: {valid_acc:.4f}, Test: {test_acc:.4f}")

# Part 2.2 — Grid Search for Best Decision Tree

param_grid = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
    'random_state': [101]
}

dt_gs = GridSearchCV(
    DecisionTreeClassifier(),
    param_grid,
    scoring='balanced_accuracy',
    cv=mysplitter,
    return_train_score=True,
    refit=False
)

dt_gs.fit(X_all, y_all)

best_idx = np.argmax(dt_gs.cv_results_['mean_test_score'])
best_params = {
    'max_depth': dt_gs.cv_results_['param_max_depth'][best_idx],
    'min_samples_leaf': dt_gs.cv_results_['param_min_samples_leaf'][best_idx],
}

print("\nBest Decision Tree hyperparameters:", best_params)

dt_best = DecisionTreeClassifier(**best_params, random_state=101)
dt_best.fit(X_train, y_train)

train_acc_best = balanced_accuracy_score(y_train, dt_best.predict(X_train))
valid_acc_best = balanced_accuracy_score(y_valid, dt_best.predict(X_valid))
test_acc_best  = balanced_accuracy_score(y_test, dt_best.predict(X_test))

# Part 3.1 — Simple Random Forest

rf_simple = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=101
)
rf_simple.fit(X_train, y_train)

train_acc_rf = balanced_accuracy_score(y_train, rf_simple.predict(X_train))
valid_acc_rf = balanced_accuracy_score(y_valid, rf_simple.predict(X_valid))
test_acc_rf  = balanced_accuracy_score(y_test, rf_simple.predict(X_test))

# Analyze feature importance
importances = rf_simple.feature_importances_
importance_df = pd.DataFrame({
    'word': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

top10 = importance_df.head(10)
near_zero = importance_df[importance_df['importance'] < 1e-5]
random_zero = near_zero.sample(min(10, len(near_zero)), random_state=42)

print("\n=== Top 10 Important Features ===")
print(top10)

print("\n=== 10 Random Low-Importance Features ===")
print(random_zero)
print(f"\nTotal features with near-zero importance (<1e-5): {len(near_zero)}")

# Part 3.2 — Hyperparameter tuning for Random Forest

param_grid_rf = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
    'min_samples_leaf': [1],
    'n_estimators': [100],
    'random_state': [101]
}

rf_gs = GridSearchCV(
    RandomForestClassifier(),
    param_grid_rf,
    scoring='balanced_accuracy',
    cv=mysplitter,
    return_train_score=True,
    refit=False
)
rf_gs.fit(X_all, y_all)

best_idx_rf = np.argmax(rf_gs.cv_results_['mean_test_score'])
best_rf_params = {
    'max_features': rf_gs.cv_results_['param_max_features'][best_idx_rf],
    'max_depth': rf_gs.cv_results_['param_max_depth'][best_idx_rf]
}
print("\nBest Random Forest hyperparameters:", best_rf_params)

rf_best = RandomForestClassifier(**best_rf_params, n_estimators=100, random_state=101)
rf_best.fit(X_train, y_train)

train_acc_rf_best = balanced_accuracy_score(y_train, rf_best.predict(X_train))
valid_acc_rf_best = balanced_accuracy_score(y_valid, rf_best.predict(X_valid))
test_acc_rf_best  = balanced_accuracy_score(y_test, rf_best.predict(X_test))

# Part 3.3 — Final Model Comparison Table

summary = pd.DataFrame([
    ["Simple DT", 1, 3, train_acc, valid_acc, test_acc],
    ["Best DT", 1, best_params['max_depth'], train_acc_best, valid_acc_best, test_acc_best],
    ["Simple RF", 100, 3, train_acc_rf, valid_acc_rf, test_acc_rf],
    ["Best RF", 100, best_rf_params['max_depth'], train_acc_rf_best, valid_acc_rf_best, test_acc_rf_best]
], columns=["Model Type", "n_trees", "max_depth", "Train Acc", "Valid Acc", "Test Acc"])

print("\n=== Final Model Comparison ===")
print(summary)

Data loaded successfully!
Train: (6346, 7729), Valid: (792, 7729), Test: (793, 7729)

=== Decision Tree (max_depth=3) ===
The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
